# PR21 · Audit the complete processing graph and its evidence

<!-- paper-first -->
### Research question

**Reading:** [PP01](../../curriculum/papers/processing.md#pp01). Review the assigned figure or result before starting the lesson.

**Question:** What would another researcher need to inspect every step between a raw image and a derivative?

Record a prediction, a source location, and one point you want this lesson to clarify. Ask your AI tutor to distinguish the paper’s evidence from its interpretation.
<!-- /paper-first -->

**Format:** 75–100 minutes for this lesson and its small executable lab, followed by the explicitly labeled upstream practice. Run this notebook from a fresh kernel, top to bottom. Core data are synthetic. Software: NumPy, SciPy, Matplotlib; extra imports are stated in code. This is one stage of a longer processing course, not a replacement for supervised research training.

## Learning objectives

- Define a workflow’s dependencies and immutable inputs.
- Build a cache key that changes when data or parameters change.
- Separate executed results, report observations and proposed external commands.

## Understand the operation

A workflow is a graph of dependencies connecting inputs, operations, derived files and quality-control decisions. The graph makes prerequisites visible: a label extraction step depends on a valid mask, compatible geometry and documented transformations. A workflow manager can schedule, cache and record these operations, but it cannot certify that the scientific choices are appropriate. A clean exit code establishes less than an inspected report.

Provenance needs enough information to reproduce the actual computation: source identifiers and hashes, software versions, command/settings, coordinate spaces, transform direction/order, interpolation, sample masks, and downstream modeling choices. A text description generated by an AI is a proposed summary until checked against those records. A cache that keys only on the input image can incorrectly reuse an old derivative when smoothing width or software changes. Content identity is useful, but a checksum cannot prove correct acquisition metadata or anatomical validity.

fMRIPrep and Nipype expose complex processing graphs and visual reports. Preprocessing outputs are not automatically the final denoised dataset for every downstream question. Confound tables supply candidates; smoothing, temporal modeling, censoring and statistical design require an explicit plan. Inspect the exact pipeline version’s behavior rather than copying an old command from a course notebook unchanged.

The offline core builds a tiny dependency graph, rejects a cycle, executes a specified smoothing step and demonstrates parameter-aware caching. The real-tool assignment then asks for a report-based audit with named evidence and unresolved findings. This capstone of the strand requires an honest boundary: none of the toy notebooks ran full FreeSurfer, FSL, ANTs, DIPY or fMRIPrep processing on a participant.

## Transformation contract

**Input:** original array, metadata, parameters, software identity and dependencies. **Output:** derivative plus provenance record and QC decision. **Preserved:** traceable linkage when all records remain available. **Not guaranteed:** correctness of the input or scientific interpretation. **Cache rule:** changes in relevant inputs/settings must invalidate reuse.


## Read the actual course material

- [Nipype tutorial: basic workflow](https://github.com/nipy/nipype_tutorial/blob/f11c9c7b8e7a1983918f1d517ec9cf3dfcb78236/notebooks/basic_workflow.ipynb). BSD-3-Clause; unmodified upstream copy in third_party/processing.
- [Nipype tutorial: BIDS inputs](https://github.com/nipy/nipype_tutorial/blob/f11c9c7b8e7a1983918f1d517ec9cf3dfcb78236/notebooks/basic_data_input_bids.ipynb). BSD-3-Clause; unmodified upstream copy in third_party/processing.
- [fMRIPrep: pipeline details](https://fmriprep.org/en/stable/workflows.html). Official project documentation; linked only.
- [fMRIPrep: public sample QC report](https://fmriprep.org/en/stable/_static/SampleReport/sample_report.html). Public report link; no image or participant data copied.

Read the named topic alongside this lesson; compare its real-image assumptions with our controlled example. These notebooks use original explanations and original code, not copied upstream passages. The source chapter is the place to continue to a complete real-tool practical. External software and downloaded datasets are not silently run by this notebook.


## Predict, then ask your AI assistant

Use Goose with your installed Ollama model, or ChatGPT. The model is a tutor and code author; the local Python runtime performs these calculations. Paste:

> Explain every graph edge and identify which scientific decisions still need a human. Build a provenance record for the actually executed SciPy step and show cache invalidation when sigma changes. Never report an external package as executed because its documentation was read. Return at most 20 executable lines per cell, show units and array shapes, and preserve the original. Explain the prediction before running. If an assertion fails, diagnose the disagreement rather than deleting the check.

Write your prediction before executing the reference cells below.


In [1]:
import numpy as np
from scipy.ndimage import gaussian_filter
from graphlib import TopologicalSorter,CycleError
import hashlib,json,platform,scipy
rng=np.random.default_rng(2121); image=rng.normal(size=(30,30))
graph={'load':set(),'bias':{'load'},'mask':{'bias'},'register':{'mask'},'inspect':{'register'},'extract':{'inspect'}}
order=list(TopologicalSorter(graph).static_order())
assert order.index('load')<order.index('register')<order.index('extract')
print('dependency order:',order)
try:
    list(TopologicalSorter({'a':{'b'},'b':{'a'}}).static_order())
except CycleError:
    print('Cycle correctly rejected.')
else:
    raise AssertionError('Dependency cycle was not detected')


dependency order: ['load', 'bias', 'mask', 'register', 'inspect', 'extract']
Cycle correctly rejected.


In [2]:
def key(array,params):
    info=json.dumps({'shape':array.shape,'dtype':str(array.dtype),'params':params,
                     'scipy':scipy.__version__},sort_keys=True).encode()
    return hashlib.sha256(info+array.tobytes()).hexdigest()
p1={'sigma_vox':1.,'mode':'reflect'}; p2={'sigma_vox':2.,'mode':'reflect'}
out1=gaussian_filter(image,p1['sigma_vox'],mode=p1['mode'])
out2=gaussian_filter(image,p2['sigma_vox'],mode=p2['mode'])
assert key(image,p1)!=key(image,p2) and not np.allclose(out1,out2)
input_only=hashlib.sha256(image.tobytes()).hexdigest()
print('Bad input-only key cannot distinguish sigma1 from sigma2:',input_only[:12])
record={'input_and_settings_hash':key(image,p1),'python':platform.python_version(),
        'numpy':np.__version__,'scipy':scipy.__version__,'parameters':p1,
        'data_kind':'synthetic','operation_executed':'SciPy Gaussian filter only',
        'fmriprep_executed':False,'output_sha256':hashlib.sha256(out1.tobytes()).hexdigest()}
print(json.dumps(record,indent=2))


Bad input-only key cannot distinguish sigma1 from sigma2: 986c2395dfc8
{
  "input_and_settings_hash": "3d38e843c634fb980b22bc4fae8fcf003fcf8e75354f862b50f98ef268eae0e9",
  "python": "3.14.7",
  "numpy": "2.5.3",
  "scipy": "1.18.1",
  "parameters": {
    "sigma_vox": 1.0,
    "mode": "reflect"
  },
  "data_kind": "synthetic",
  "operation_executed": "SciPy Gaussian filter only",
  "fmriprep_executed": false,
  "output_sha256": "cfcf71212fc846cf022af07adb262d3234356b9a82fda88f72124d7c192add1a"
}


## Check and explain

The ordering respects dependencies, cycles are rejected, and changing sigma changes both the derivative and parameter-aware key. The record explicitly identifies synthetic data and states fMRIPrep was not executed. Its hashes do not replace a QC assessment.

## Deliberately wrong method

An input-only cache silently reuses results across different parameters. An AI-generated methods paragraph that names tools never run is another provenance failure. Both can look polished while contradicting the actual computation.

## Transfer to an actual dataset or tool — guided assignment

Open the vendored Nipype basic-workflow and preprocessing notebooks as reading material; their older dependencies and data are separate. Trace their graph to the fMRIPrep pipeline diagram. Inspect the official sample report: record its stated version, first-run TR, distortion status, mask/alignment panels and missing information. For an approved real project, retain a manifest, report and decision log. Mark every external practical not yet run and require mentor review before a biological claim. This is the strand’s audit portfolio.

**Submit:** a transformation card, one labeled figure or numerical result, the failed-method diagnosis, and the upstream-practice evidence. If the external exercise has not been run, mark it **not executed** and state the missing software/data; do not convert a proposed command into a claimed result.

## Exit questions and answer key

1. Why include parameters in a cache key? **Answer:** identical inputs under different operations produce different derivatives.
2. What proves a tool ran? **Answer:** an actual execution record and outputs tied to its inputs/version, not a suggested command or a source citation.


### Return to the research question

Revisit [PP01](../../curriculum/papers/processing.md#pp01) and your initial prediction. In your [evidence ledger](../../curriculum/coursework/EVIDENCE_LEDGER.md):

1. Cite one output or diagnostic from this lesson and explain the transformation it demonstrates.
2. Revise one claim or question from the paper, with a figure or section locator. Which part of the published result remains open after this exercise?
3. Ask AI to propose a next check. Accept, revise or reject it with a scientific reason. Then explain your decision aloud without reading the AI response.

Include this entry in the A2 portfolio when relevant.
